# Overview
Author: Darrin O'Brien, darrinobrien5@gmail.com

1. Fine-Tunes ResNet 50 on the MNIST train dataset purely on vision (discarded text encoder). Adds a Classifier Head that is fine-tuned along with the visual encoder.
2. Evaluates Fine-Tuned vs Base ResNet50 on MNIST test dataset. 

Paper for MNIST: https://arxiv.org/abs/1512.03385

## Installations

In [ ]:
!pip install torch torchvision
!pip install -U datasets transformers
!pip install tqdm 
!pip install -U Pillow
!pip install numpy==1.26.4

### Runpod Installs Only

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
# !pip uninstall -y Pillow
# !pip install Pillow

## Imports

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from datasets import load_dataset, load_from_disk, Dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image, ExifTags
from tqdm import tqdm
import numpy as np

## Model and Dataset Prep

In [ ]:
device = "cuda" if torch.cuda_is_available() else "cpu"

# https://huggingface.co/datasets/ylecun/mnist. 10 classes total {0,1,2,3,...9}.
dataset = load_dataset("ylecun/mnist")
split = dataset["train"].train_test_split(test_size=0.2, seed=66)

train = split["train"]
val = split["test"]
test = dataset["test"]

processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model = AutoModelForImageClassification.from_pretrained("microsoft/resnet-50", torch_dtype=torch.float32).to(device)

In [ ]:
def processImage(example):
    img = processor(example["image"], return_tensors="np")
    example["image"] = img["image"][0]
    return example

train = train.map(processImage)
val = val.map(processImage)
test = test.map(processImage)

In [ ]:
train.save_to_disk("/workspace/preprocessed/MNIST/train_converted")
val.save_to_disk("/workspace/preprocessed/MNIST/val_converted")
test.save_to_disk("/workspace/preprocessed/MNIST/test_converted")

In [ ]:
train = load_from_disk("/workspace/preprocessed/MNIST/train_converted")
val = load_from_disk("/workspace/preprocessed/MNIST/val_converted")
test = load_from_disk("/workspace/preprocessed/MNIST/test_converted")

In [ ]:
class ClassifierHead(nn.Module):
    def __init__(self, model, num_labels=10):
        self.ResNet50 = model
        self.num_labels = num_labels
        self.classifierHead = nn.Linear(model.named_parameters(), num_labels)
    
    def forward(self, images):
        image_features =  self.ResNet50(images)
        logits = self.classifierHead(image_features)
        return logits

model = ClassifierHead(model).to(device)

## Fine-Tuning Prep

In [ ]:
for parameter in model.named_parameters():
    parameter.requires_grad_(True)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {'params': model.named_parameters(), 'lr': 1e-5},
    {'params': model.classifier.named_parameters(), 'lr': 1e-3}
], weight_decay=0.01)

EPOCHS = 10

# Scheduler Next